In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import numpy as np

In [25]:
df = pd.read_csv("orders.csv", sep=';')


In [ ]:

# Проверяем размер
print("Исходный размер:", df.shape)
popular_products = [
    "Banana",
    "Soap",
    "Organic Strawberries",
    "Organic Baby Spinach",
    "Organic Hass Avocado",
    "Organic Avocado",
    "Large Lemon",
    "Organic Raspberries",
    "Organic Whole Milk",
    "Strawberries",
    "Limes",
    "Organic Garlic",
    "Organic Zucchini",
    "Organic Yellow Onion",
    "Cucumber Kirby",
    "Organic Blueberries",
    "Organic Fuji Apple",
    "Apple Honeycrisp Organic",
    "Organic Lemon",
    "Seedless Red Grapes",
    "Sparkling Water Grapefruit",
    "Yellow Onions",
    "Organic Baby Carrots",
    "Organic Baby Arugula",
    "Organic Grape Tomatoes",
    "Honeycrisp Apple",
    "Organic Half & Half",
    "Organic Cucumber",
    "Organic Small Bunch Celery",
    "Organic Large Extra Fancy Fuji Apple",
    "Carrots",
    "Original Hummus",
    "Organic Gala Apples",
    "Fresh Cauliflower",
    "Michigan Organic Kale",
    "Organic Red Onion",
    "Organic Blackberries",
    "Organic Cilantro",
    "Spring Water",
    "Half & Half",
    "Asparagus",
    "100% Whole Wheat Bread",
    "Raspberries",
    "Organic Italian Parsley Bunch",
    "Organic Unsweetened Almond Milk",
    "Organic Tomato Cluster",
    "Organic Whole String Cheese",
    "Organic Red Bell Pepper",
    "Red Vine Tomato"
]

#df_filtered = df[df['product_name'].isin(popular_products)]
#print("Исходный фильтр размер:", df_filtered.shape)
df = df.sample(n=1_000_00, random_state=42)

def replace_unpopular_products(group):
    items = group['product_name'].tolist()
    new_items = []
    for item in items:
        if item in popular_products:
            new_items.append(item)
        else:
            # выбираем случайный популярный товар (можно с повторением)
            new_item = np.random.choice(popular_products)
            new_items.append(new_item)
    group['product_name'] = new_items
    return group

# Применяем по каждому заказу
df_modified = df.groupby('order_id').apply(replace_unpopular_products).reset_index(drop=True)
print(1)
def remove_duplicates(group):
    # оставляем только уникальные продукты
    group = group.drop_duplicates(subset=['product_name'])
    return group

df_final = df_modified.groupby('order_id').apply(remove_duplicates).reset_index(drop=True)
print(df_modified.shape)
  # фиксируем random_state для воспроизводимости


# Сохраняем уменьшенный CSV
df_final.to_csv("orders_products_sample.csv", index=False)

In [45]:
df = pd.read_csv("orders.csv")



In [38]:
product_counts = df['product_name'].value_counts()
# frequent_products = product_counts[product_counts >= 150].index
# df = df[df['product_name'].isin(frequent_products)]
# print(df.shape)
# print(df['product_name'].value_counts()/len(df) * 100)
product_counts


product_name
Olive Oil, Salmon Fillet, Lemon                                                                               1025
Olive Oil, Lemon                                                                                               502
Salmon Fillet, Lemon                                                                                           482
Olive Oil, Salmon Fillet                                                                                       457
Baby Carrots, Cucumber, Hummus                                                                                 233
                                                                                                              ... 
Yellow Onion, Soap, Grape Tomatoes, Zucchini, Red Bell Pepper, Asparagus, Tomato Cluster, Honeycrisp Apple       1
Lemon, Baby Spinach, Baby Carrots, Fuji Apple, Banana, Avocado, Honeycrisp Apple                                 1
Red Bell Pepper, Zucchini, Salmon Fillet, Lemon                    

In [46]:
basket = df['product_name'].apply(lambda x: x.split(','))
print(basket.head())

0     [Salmon Fillet,  Olive Oil,  Whole Milk,  Lemon]
1    [Baby Carrots,  Cucumber,  Cauliflower,  Vine ...
2                  [Olive Oil,  Salmon Fillet,  Lemon]
3    [Soap,  Blueberries,  Still Water,  Strawberri...
4                        [Carrots,  Cucumber,  Hummus]
Name: product_name, dtype: object


In [47]:
te = TransactionEncoder()
te_array = te.fit(basket).transform(basket)
basket = pd.DataFrame(te_array, columns=te.columns_)
basket


,Almond Milk,Asparagus,Avocado,Baby Carrots,Baby Spinach,Banana,Blackberries,Blueberries,Carrots,Cauliflower,...,Sparkling Water,Still Water,Strawberries,String Cheese,Tomato Cluster,Vine Tomato,Whole Milk,Whole Wheat Bread,Yellow Onion,Zucchini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False
149996,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
149997,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
149998,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [48]:
frequent = apriori(basket, min_support=0.01, use_colnames=True)

print("Частых наборов:", len(frequent))
display(frequent.head())

Частых наборов: 301


,support,itemsets
0,0.060380,( Almond Milk)
1,0.060027,( Asparagus)
2,0.063120,( Avocado)
3,0.062753,( Baby Carrots)
4,0.096333,( Baby Spinach)


In [49]:
rules = association_rules(frequent, metric="confidence", min_threshold=0.01)

rules = rules[['antecedents','consequents','support','confidence','lift']]
rules['antecedents'] = rules['antecedents'].apply(lambda x: ", ".join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ", ".join(list(x)))

print("Количество правил:", len(rules))
display(rules)

Количество правил: 628


,antecedents,consequents,support,confidence,lift
0,Asparagus,Banana,0.010353,0.172479,1.128739
1,Banana,Asparagus,0.010353,0.067754,1.128739
2,Banana,Avocado,0.010973,0.071812,1.137704
3,Avocado,Banana,0.010973,0.173849,1.137704
4,Carrots,Baby Carrots,0.010680,0.146448,2.333716
...,...,...,...,...,...
623,"String Cheese, Whole Milk",Whole Wheat Bread,0.012140,0.480983,6.766778
624,"Whole Wheat Bread, Whole Milk",String Cheese,0.012140,0.544720,4.400476
625,String Cheese,"Whole Wheat Bread, Whole Milk",0.012140,0.098072,4.400476
626,Whole Wheat Bread,"String Cheese, Whole Milk",0.012140,0.170793,6.766778


In [50]:
def simplify(itemset):
    return tuple(sorted(list(itemset)))

rules['A'] = rules['antecedents'].apply(simplify)
rules['C'] = rules['consequents'].apply(simplify)

# Для удаления дублей считаем объединённое множество
rules['union'] = rules.apply(lambda x: tuple(sorted(set(x['A']) | set(x['C']))), axis=1)

# Убираем дубли: одно union → одно правило с максимальной lift
rules_cleaned = rules.sort_values('lift', ascending=False).drop_duplicates('union')

# Чистим временные колонки
rules_cleaned = rules_cleaned.drop(columns=['A','C','union'])



print("Количество правил после очистки:", len(rules_cleaned))


Количество правил после очистки: 203


In [51]:
# Объединяем antecedents и consequents в один столбец
def merge_rule(row):
    merged = sorted(list(set(row['antecedents'].split(",")) | set(row['consequents'].split(","))))
    return ", ".join(merged)

rules_cleaned["rule_combined"] = rules_cleaned.apply(merge_rule, axis=1)

# Оставим только один столбец, если нужны только объединённые правила
rules_final = rules_cleaned.drop(columns=['antecedents','consequents'])

print("Количество уникальных объединённых правил:", len(rules_final))
display(rules_final)


Количество уникальных объединённых правил: 203


,support,confidence,lift,rule_combined
557,0.012953,0.603416,10.567708,"Lemon, Salmon Fillet, Olive Oil"
505,0.010453,0.221908,9.939134,"Fresh Basil, Olive Oil, Pasta Penne"
571,0.010473,0.222332,9.555829,"Mozzarella Cheese, Olive Oil, Pasta Penne"
492,0.011527,0.395833,8.402915,"Mozzarella Cheese, Fresh Basil, Pasta Penne"
510,0.010293,0.367444,7.800256,"Tomato Cluster, Fresh Basil, Pasta Penne"
...,...,...,...,...
361,0.010660,0.086381,0.697822,"String Cheese, Tomato Cluster"
48,0.010153,0.104222,0.682053,"Banana, Hummus"
270,0.011313,0.081072,0.656950,"Lemon, Tomato Cluster"
219,0.010380,0.074384,0.630691,"Greek Yogurt, Lemon"


In [52]:
def get_recommendations(item, metric, rules, top_n=5):
    """
    Возвращает рекомендации для товара item на основе правил ассоциаций.

    Параметры:
        item (str): товар, для которого ищутся рекомендации
        metric (str): метрика сортировки ('confidence', 'lift', 'support')
        rules (DataFrame): таблица правил
        top_n (int): количество рекомендаций
    
    Возвращает:
        DataFrame: top_n рекомендаций
    """

    # Проверяем корректность метрики
    allowed = ["support", "confidence", "lift"]
    if metric not in allowed:
        raise ValueError(f"metric должен быть одним из: {allowed}")

    df = rules.copy()

    # Разбиваем строку "Banana, Limes" → ["Banana", "Limes"]
    df["items_list"] = df["rule_combined"].apply(lambda x: [i.strip() for i in x.split(",")])

    # Оставляем только те строки, где item присутствует
    df = df[df["items_list"].apply(lambda x: item in x)]

    if df.empty:
        print(f"Нет рекомендаций для товара: {item}")
        return pd.DataFrame()

    # Рекомендации = все товары кроме item
    df["recommendation"] = df["items_list"].apply(
        lambda lst: [p for p in lst if p != item]
    )

    # Каждая строка может содержать несколько рекомендаций → распаковываем
    df = df.explode("recommendation")

    # Удаляем строки-пустышки
    df = df[df["recommendation"].notna()]

    # Сортируем по метрике
    df = df.sort_values(metric, ascending=False)

    return df[["recommendation", "support", "confidence", "lift"]].head(top_n).reset_index(drop=True)

In [54]:
get_recommendations("Lemon", "lift", rules_final, top_n=10)

,recommendation,support,confidence,lift
0,Salmon Fillet,0.012953,0.603416,10.567708
1,Olive Oil,0.012953,0.603416,10.567708
2,Salmon Fillet,0.025327,0.443549,3.178502
3,Olive Oil,0.021467,0.153831,3.084031
4,Garlic,0.010667,0.076438,0.980813
5,Red Bell Pepper,0.010613,0.076056,0.919511
6,Zucchini,0.010980,0.078683,0.870199
7,Cauliflower,0.010713,0.076772,0.857154
8,Vine Tomato,0.011473,0.116394,0.834086
9,String Cheese,0.013727,0.098366,0.794642


In [55]:
get_recommendations("Lemon", "confidence", rules_final, top_n=10)

,recommendation,support,confidence,lift
0,Salmon Fillet,0.012953,0.603416,10.567708
1,Olive Oil,0.012953,0.603416,10.567708
2,Salmon Fillet,0.025327,0.443549,3.178502
3,Olive Oil,0.021467,0.153831,3.084031
4,Vine Tomato,0.011473,0.116394,0.834086
5,Mozzarella Cheese,0.014593,0.104577,0.732638
6,Kale,0.011447,0.102888,0.737304
7,String Cheese,0.013727,0.098366,0.794642
8,Banana,0.014920,0.097640,0.699692
9,Fuji Apple,0.012000,0.085993,0.770406
